In [1]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of the united"
tokens = u_tokenizer.encode(prompt)
tokens

tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3])

In [2]:
context_length = 32

In [44]:
torch.manual_seed(1)
u_model = MasterModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=12, num_heads=4, context_length=context_length, num_layers=8)

out = u_model(tokens)
out.shape

torch.Size([14, 64])

In [45]:
u_model

MasterModel(
  (embedding): Embedding(64, 12)
  (pos_embedding): Embedding(32, 12)
  (layers): Sequential(
    (0): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): LayerNorm()
    )
    (1): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_fea

In [5]:
with open("text.txt", "r") as f:
    text = f.read()
text[:100], len(text)

('the capital of the united states is not london. the capital of france is paris, and berlin is the ca',
 4099)

In [ ]:
token_ids = u_tokenizer.encode(text)
len(token_ids)

1593

In [7]:
token_ids

tensor([ 0, 61,  1,  ..., 61, 37, 59])

In [8]:
ids = token_ids.detach().cpu().numpy().tolist()
type(ids)

list

In [9]:
from text_dataset import TextDataset

stride = 12
dataset = TextDataset(ids, stride=stride, context_length=context_length)
dataset.inputs[0]

tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3, 61,  4, 58, 61,  5, 61,  6, 61,  7,
        59, 61,  0, 61,  1, 61,  2, 61,  8, 61,  5, 61,  9, 60])

In [10]:
len(dataset.inputs), len(dataset.targets)

(131, 131)

In [11]:
dataset.inputs[0], dataset.targets[0]

(tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3, 61,  4, 58, 61,  5, 61,  6, 61,  7,
         59, 61,  0, 61,  1, 61,  2, 61,  8, 61,  5, 61,  9, 60]),
 tensor([61,  1, 61,  2, 61,  0, 61,  3, 61,  4, 58, 61,  5, 61,  6, 61,  7, 59,
         61,  0, 61,  1, 61,  2, 61,  8, 61,  5, 61,  9, 60, 61]))

In [12]:
# model parameters count
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

12160


In [13]:
print(u_model)

MasterModel(
  (embedding): Embedding(64, 12)
  (pos_embedding): Embedding(32, 12)
  (layers): Sequential(
    (0): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): LayerNorm()
    )
    (1): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_fea

In [14]:
out0 = u_model(dataset.inputs[0])
out0.shape

torch.Size([32, 64])

In [15]:
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()


In [16]:
loss = loss_fn(out0, dataset.targets[0])
loss

tensor(4.6109, grad_fn=<NllLossBackward0>)

In [46]:
import torch
# optimizer tüm parametreleri optimize eder. duzeltilmesi gerekiyor.
optimizer = torch.optim.Adam(u_model.parameters(), lr=0.001)


In [47]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [ ]:
epochs = 1000
for epoch in range(epochs):
    total_loss = 0.0
    for X, y in dataset:
        #X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = u_model(X)
        loss = loss_fn(pred, y)
        total_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    average_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}, average_loss: {average_loss:.4f} ")

Epoch 1/1000, Loss: 2.9067, average_loss: 3.2012 
Epoch 2/1000, Loss: 2.6961, average_loss: 2.7656 
Epoch 3/1000, Loss: 2.3895, average_loss: 2.4681 
Epoch 4/1000, Loss: 2.3228, average_loss: 2.2810 
Epoch 5/1000, Loss: 2.3144, average_loss: 2.2024 
Epoch 6/1000, Loss: 2.2166, average_loss: 2.1526 
Epoch 7/1000, Loss: 2.1086, average_loss: 2.1197 
Epoch 8/1000, Loss: 2.1533, average_loss: 2.1115 
Epoch 9/1000, Loss: 2.1283, average_loss: 2.0916 
Epoch 10/1000, Loss: 2.1089, average_loss: 2.0844 
Epoch 11/1000, Loss: 2.1150, average_loss: 2.0882 
Epoch 12/1000, Loss: 2.0978, average_loss: 2.0870 
Epoch 13/1000, Loss: 2.0886, average_loss: 2.0705 
Epoch 14/1000, Loss: 2.0871, average_loss: 2.0674 
Epoch 15/1000, Loss: 2.1087, average_loss: 2.0707 
Epoch 16/1000, Loss: 2.1129, average_loss: 2.0704 
Epoch 17/1000, Loss: 2.1263, average_loss: 2.0665 
Epoch 18/1000, Loss: 2.1100, average_loss: 2.0576 
Epoch 19/1000, Loss: 2.0933, average_loss: 2.0593 
Epoch 20/1000, Loss: 2.0987, average_los

In [49]:
import torch
out = u_model(tokens)
probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index

(tensor(0.4649, grad_fn=<MaxBackward0>), tensor(5))

In [61]:
new_tokens = u_tokenizer.encode("the capital of france is paris, and berlin ")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
out = u_model(torch.tensor(new_tokens)) # 4 tokeni state bildi (:
probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index

(tensor(0.8412, grad_fn=<MaxBackward0>), tensor(0))

In [63]:
u_model.embedding.weight

Parameter containing:
tensor([[-7.9181e-01, -1.0277e-01, -4.7933e-01, -1.0459e+00,  8.4480e-01,
         -1.1121e+00, -7.2254e-01, -1.8630e+00, -9.5648e-01,  5.3345e-01,
         -8.6266e-01, -3.6351e-01],
        [ 1.0269e+00,  1.9967e+00,  1.7498e-01, -3.4023e-01, -4.6603e-01,
          2.2338e-01,  2.9877e-01,  7.7397e-01,  9.0184e-01,  6.8081e-01,
         -4.2401e-02,  9.0908e-01],
        [ 1.9833e+00, -2.5875e-01, -1.5590e+00, -2.0169e+00,  2.1019e-01,
         -2.4508e+00, -1.0160e+00, -7.1313e-01,  7.2847e-01,  4.0798e-01,
          5.6950e-01,  1.5588e-01],
        [ 1.5986e-01, -4.6246e-01,  2.0030e-01, -3.2575e-01, -1.3970e-01,
          3.7914e-01, -5.3827e-01, -1.2488e+00,  5.6907e-01, -1.2082e+00,
          1.9241e+00, -7.8610e-01],
        [-4.2004e-01, -2.0662e-01, -3.6684e-01, -2.4636e-01, -3.8146e-01,
          1.2368e+00,  8.7413e-01,  2.6620e+00, -3.9027e-01, -8.4848e-02,
          6.1890e-01, -4.5252e-01],
        [ 1.6894e+00, -8.3607e-01, -2.2452e+00, -1.1983e+0

In [64]:
# save the model
torch.save(u_model.state_dict(), "master_model.pth")

# load the model
u_model.load_state_dict(torch.load("master_model.pth"))

# generate text
new_tokens = u_tokenizer.encode("the capital of the united states is london. the capital of france is")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)


<function len(obj, /)>

In [65]:
len(new_tokens)

28

In [67]:
loaded_model = MasterModel(vocab_size=64, embedding_dim=12, num_heads=4, context_length=32, num_layers=8)
loaded_model.load_state_dict(torch.load("master_model.pth"))
loaded_model

MasterModel(
  (embedding): Embedding(64, 12)
  (pos_embedding): Embedding(32, 12)
  (layers): Sequential(
    (0): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): LayerNorm()
    )
    (1): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_fea